# Lesson 3.2: Managing long conversations

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

# 1. Summarization

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model

# 1. Main Agent Model
gemini_lite = init_chat_model(
    model="models/gemini-3.5-flash-lite", 
    model_provider="google_genai"
)

# 2. Ultra-Fast Background Summarizer (Groq)
groq_summarizer = init_chat_model(
    model="openai/gpt-oss-120b", 
    model_provider="groq"
)

# 3. Agent with hybrid summarization
agent = create_agent(
    model=gemini_lite,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=groq_summarizer, # <--- Groq compresses history in the background!
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nProvide fictional information about the moon’s capital, Lunapolis, including its weather, population of cheese miners, and the likelihood of a union strike.\n\n## SUMMARY\n- Capital of the moon: **Lunapolis**.  \n- Weather in Lunapolis: **clear skies, high of 120\u202f°C, low of –100\u202f°C**.  \n- Cheese miners population in Lunapolis: **approximately 100,000**.  \n- Prediction on cheese miners' union strike: **Yes, likely to strike due to dissatisfaction with the new president**.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone – all user queries have been answered.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='4d8b725e-a444-4c0e-92a4-5821f607dd95'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='5923c68c-c17d-4cb2-a422-44910ef

In [5]:
print(response["messages"][-1].content[0]['text'])

If I were the newly elected president of Lunapolis, facing a potential strike by 100,000 disgruntled cheese miners—and dealing with extreme surface temperatures ranging from 120 °C to –100 °C—I would take immediate, proactive steps to avert a crisis and secure our dairy supply chain. 

Here is how I would address the situation:

### 1. Convene an Emergency "Gouda-Faith" Summit
I would immediately invite the leadership of the Amalgamated Lunar Lactose & Excavation Union (ALLEU) to the presidential habitat for open negotiations. Before talking politics, I’d serve a spread of our finest locally aged cheeses to show respect for their hard work. My opening message would be simple: *Lunapolis runs on cheese, and we cannot afford to curdle our relationship.*

### 2. Address Their Core Grievances
I know the miners have valid concerns regarding their contracts, hazard pay for deep-crust excavation, and the rising cost of oxygen rationing. As president, I would propose:
* **Thermal Hazard Pay:**

# 2. Delete messages

In [6]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [7]:
gemini_lite = init_chat_model(
    model="models/gemini-3.5-flash-lite", 
    model_provider="google_genai"
)

agent = create_agent(
    model=gemini_lite,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages]
)

In [9]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='54aab89e-154b-4526-963e-4ef5a225d74a'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='4a34dcf4-7975-4ffa-9a16-b4808bcb98ac', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='e990a14e-5eb7-4804-b88f-2b2f04461cbc'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='36ee6816-ca76-45df-998f-d2e54367bdf6', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='7e07b182-0e33-4f19-8119-e296eabb6ae8'),
              HumanMessage(content="My device won't turn on. What should I do?", additional_kwa

In [10]:
print(response["messages"][-1].content[0]['text'])

To answer your question, checking the temperature is actually a very good next step! 

* **If it feels very hot:** The device might have overheated and triggered an automatic safety shutdown. Unplug it, let it cool down for 15–30 minutes, and then try turning it on again.
* **If it feels ice cold** (especially if it was left in a car in the winter): Batteries often refuse to power on when they are too cold. Let it warm up to room temperature.
* **If it feels normal (room temperature):** Then temperature isn't the issue, and we can move on to the next troubleshooting step. 

Since it's plugged in and turned on, but you haven't mentioned any lights: **Are you seeing any indicator lights (like a charging light or power light), or is it completely dead and dark?**
